# Seq2Seq + Attention Mechanism sur le dataset English-French

Ce notebook suit le process de `092` : chargement des données, sous-échantillon de 10 % du train, split interne train/validation, Optuna, entraînement final, puis évaluation avec SacreBLEU, chrF++ et METEOR.


In [5]:
%pip install --quiet tensorflow pandas scikit-learn matplotlib optuna sacrebleu nltk

Note: you may need to restart the kernel to use updated packages.


In [6]:
from pathlib import Path
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
from tensorflow import keras
from tensorflow.keras import layers
import optuna
import sacrebleu
from nltk.translate.meteor_score import meteor_score as nltk_meteor_score
import nltk

SEED = 42
tf.keras.utils.set_random_seed(SEED)
np.random.seed(SEED)

LOCAL_DATA_PATH = Path(r"D:/100DaysofML/Notebooks/093_Attention_Mechanism/english_french.csv")
TRAIN_SAMPLE_RATIO = 0.10
VAL_RATIO = 0.10
TEST_RATIO = 0.10
MAX_VOCAB = 30000
MAX_LEN_FR = 40
MAX_LEN_EN = 40
BATCH_SIZE = 64
OPTUNA_TRIALS = 3
OPTUNA_EPOCHS = 2
FINAL_EPOCHS = 6

EMBEDDING_DIMS = [128, 256]
ENCODER_UNITS = [128, 256]
DROPOUT_RANGE = (0.10, 0.35)
LEARNING_RATE_RANGE = (1e-4, 3e-3)

START_TOKEN = "sostok"
END_TOKEN = "eostok"

nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)
nltk.download("punkt", quiet=True)

True

In [7]:
def load_translation_csv_splits(csv_path: Path):
    if not csv_path.exists():
        raise FileNotFoundError(f"CSV file not found: {csv_path}")

    dataset_df = pd.read_csv(csv_path).dropna(subset=["English", "French"]).copy()
    dataset_df["English"] = dataset_df["English"].astype(str).str.strip()
    dataset_df["French"] = dataset_df["French"].astype(str).str.strip()

    train_df, temp_df = train_test_split(
        dataset_df,
        test_size=VAL_RATIO + TEST_RATIO,
        random_state=SEED,
        shuffle=True,
    )
    val_df, test_df = train_test_split(
        temp_df,
        test_size=0.5,
        random_state=SEED,
        shuffle=True,
    )
    return train_df.reset_index(drop=True), val_df.reset_index(drop=True), test_df.reset_index(drop=True)


train_dataset, val_dataset, test_dataset = load_translation_csv_splits(LOCAL_DATA_PATH)

# Le 10 % est prélevé ici sur le train complet, avant le split interne train/validation.
train_shuffled = train_dataset.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
sample_size = max(1, int(len(train_shuffled) * TRAIN_SAMPLE_RATIO))
sample_size = min(sample_size, len(train_shuffled))
sampled_train = train_shuffled.iloc[:sample_size].reset_index(drop=True)
train_dataset, optuna_val_dataset = train_test_split(
    sampled_train,
    test_size=VAL_RATIO,
    random_state=SEED,
    shuffle=True,
)
train_dataset = train_dataset.reset_index(drop=True)
optuna_val_dataset = optuna_val_dataset.reset_index(drop=True)
official_val_dataset = val_dataset
official_test_dataset = test_dataset


def extract_pairs(split_dataset):
    source_texts = split_dataset["French"].astype(str).tolist()
    target_texts = [f"{START_TOKEN} {text} {END_TOKEN}" for text in split_dataset["English"].astype(str).tolist()]
    return source_texts, target_texts


train_sources, train_targets = extract_pairs(train_dataset)
optuna_val_sources, optuna_val_targets = extract_pairs(optuna_val_dataset)
official_val_sources, official_val_targets = extract_pairs(official_val_dataset)
official_test_sources, official_test_targets = extract_pairs(official_test_dataset)

print("Train size:", len(train_sources))
print("Optuna val size:", len(optuna_val_sources))
print("Official val size:", len(official_val_sources))
print("Official test size:", len(official_test_sources))

Train size: 16545
Optuna val size: 1839
Official val size: 22980
Official test size: 22981


In [8]:
source_vectorizer = keras.layers.TextVectorization(
    max_tokens=MAX_VOCAB,
    standardize="lower_and_strip_punctuation",
    split="whitespace",
    output_mode="int",
    output_sequence_length=MAX_LEN_FR,
)

target_vectorizer = keras.layers.TextVectorization(
    max_tokens=MAX_VOCAB,
    standardize="lower_and_strip_punctuation",
    split="whitespace",
    output_mode="int",
    output_sequence_length=MAX_LEN_EN + 2,
)

source_vectorizer.adapt(train_sources)
target_vectorizer.adapt(train_targets)


def vectorize_inputs(source_texts, target_texts):
    source_tokens = source_vectorizer(np.array(source_texts)).numpy().astype("int32")
    target_tokens = target_vectorizer(np.array(target_texts)).numpy().astype("int32")
    decoder_inputs = target_tokens[:, :-1]
    decoder_targets = target_tokens[:, 1:]
    sample_weights = (decoder_targets != 0).astype("float32")
    return source_tokens, decoder_inputs, decoder_targets, sample_weights


X_train, Y_train_in, Y_train_out, W_train = vectorize_inputs(train_sources, train_targets)
X_optuna_val, Y_optuna_val_in, Y_optuna_val_out, W_optuna_val = vectorize_inputs(optuna_val_sources, optuna_val_targets)
X_official_val, Y_official_val_in, Y_official_val_out, W_official_val = vectorize_inputs(official_val_sources, official_val_targets)
X_official_test, Y_official_test_in, Y_official_test_out, W_official_test = vectorize_inputs(official_test_sources, official_test_targets)

print("X_train shape:", X_train.shape)
print("Y_train_in shape:", Y_train_in.shape)
print("Y_train_out shape:", Y_train_out.shape)
print("X_optuna_val shape:", X_optuna_val.shape)
print("X_official_test shape:", X_official_test.shape)

X_train shape: (16545, 40)
Y_train_in shape: (16545, 41)
Y_train_out shape: (16545, 41)
X_optuna_val shape: (1839, 40)
X_official_test shape: (22981, 40)


In [ ]:
def build_seq2seq_attention_model(
    src_vocab_size,
    tgt_vocab_size,
    embedding_dim=128,
    encoder_units=256,
    decoder_units=256,
    dropout=0.2,
    learning_rate=1e-3,
):
    encoder_inputs = keras.Input(shape=(MAX_LEN_FR,), dtype="int32", name="encoder_inputs")
    encoder_embedding = layers.Embedding(src_vocab_size, embedding_dim, mask_zero=True, name="encoder_embedding")
    encoder_x = encoder_embedding(encoder_inputs)
    encoder_outputs, encoder_state = layers.GRU(
        encoder_units,
        return_sequences=True,
        return_state=True,
        dropout=dropout,
        implementation="2"
        name="encoder_gru",
    )(encoder_x)

    decoder_inputs = keras.Input(shape=(MAX_LEN_EN + 1,), dtype="int32", name="decoder_inputs")
    decoder_embedding = layers.Embedding(tgt_vocab_size, embedding_dim, mask_zero=True, name="decoder_embedding")
    decoder_x = decoder_embedding(decoder_inputs)
    decoder_outputs, _ = layers.GRU(
        decoder_units,
        return_sequences=True,
        return_state=True,
        dropout=dropout,
        implementation="2"
        name="decoder_gru",
    )(decoder_x, initial_state=encoder_state)

    attention = layers.AdditiveAttention(name="attention")
    context = attention([decoder_outputs, encoder_outputs])
    decoder_context = layers.Concatenate(name="decoder_attention_concat")([decoder_outputs, context])
    decoder_context = layers.Dropout(dropout, name="decoder_dropout")(decoder_context)
    outputs = layers.Dense(tgt_vocab_size, activation="softmax", name="decoder_dense")(decoder_context)

    model = keras.Model([encoder_inputs, decoder_inputs], outputs, name="seq2seq_attention_english_french")
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss="sparse_categorical_crossentropy",
)
    return model


def build_inference_models(model):
    encoder_inputs = model.inputs[0]
    encoder_embedding = model.get_layer("encoder_embedding")
    encoder_gru = model.get_layer("encoder_gru")
    attention = model.get_layer("attention")
    decoder_embedding = model.get_layer("decoder_embedding")
    decoder_gru = model.get_layer("decoder_gru")
    decoder_dense = model.get_layer("decoder_dense")

    encoder_x = encoder_embedding(encoder_inputs)
    encoder_outputs, encoder_state = encoder_gru(encoder_x)
    encoder_model = keras.Model(encoder_inputs, [encoder_outputs, encoder_state], name="encoder_model")

    encoder_outputs_input = keras.Input(shape=(MAX_LEN_FR, encoder_gru.units), name="encoder_outputs_input")
    decoder_state_input = keras.Input(shape=(decoder_gru.units,), name="decoder_state_input")
    decoder_step_input = keras.Input(shape=(1,), dtype="int32", name="decoder_step_input")

    decoder_x = decoder_embedding(decoder_step_input)
    decoder_outputs, decoder_state = decoder_gru(decoder_x, initial_state=decoder_state_input)
    context = attention([decoder_outputs, encoder_outputs_input])
    decoder_context = layers.Concatenate(name="decoder_attention_concat_inference")([decoder_outputs, context])
    logits = decoder_dense(decoder_context)

    decoder_model = keras.Model(
        [decoder_step_input, encoder_outputs_input, decoder_state_input],
        [logits, decoder_state],
        name="decoder_model",
    )
    return encoder_model, decoder_model


def make_greedy_decoder(encoder_model, decoder_model, target_vectorizer):
    vocab = target_vectorizer.get_vocabulary()
    token_to_index = {token: index for index, token in enumerate(vocab)}
    start_index = token_to_index[START_TOKEN]
    end_index = token_to_index[END_TOKEN]

    def decode_sequence(source_text):
        source_tokens = source_vectorizer(np.array([source_text])).numpy().astype("int32")
        encoder_outputs, state = encoder_model.predict(source_tokens, verbose=0)
        current_token = np.array([[start_index]], dtype="int32")
        decoded_tokens = []

        for _ in range(MAX_LEN_EN + 1):
            token_probs, state = decoder_model.predict([current_token, encoder_outputs, state], verbose=0)
            next_index = int(np.argmax(token_probs[0, 0]))
            if next_index == 0 or next_index == end_index:
                break
            decoded_tokens.append(vocab[next_index])
            current_token = np.array([[next_index]], dtype="int32")

        return " ".join(decoded_tokens)

    return decode_sequence


source_vocab_size = len(source_vectorizer.get_vocabulary())
target_vocab_size = len(target_vectorizer.get_vocabulary())
print("Source vocab size:", source_vocab_size)
print("Target vocab size:", target_vocab_size)

Source vocab size: 10901
Target vocab size: 6097


In [11]:
def objective(trial):
    embedding_dim = trial.suggest_categorical("embedding_dim", EMBEDDING_DIMS)
    encoder_units = trial.suggest_categorical("encoder_units", ENCODER_UNITS)
    dropout = trial.suggest_float("dropout", DROPOUT_RANGE[0], DROPOUT_RANGE[1])
    learning_rate = trial.suggest_float("learning_rate", LEARNING_RATE_RANGE[0], LEARNING_RATE_RANGE[1], log=True)

    model = build_seq2seq_attention_model(
        source_vocab_size,
        target_vocab_size,
        embedding_dim=embedding_dim,
        encoder_units=encoder_units,
        decoder_units=encoder_units,
        dropout=dropout,
        learning_rate=learning_rate,
)

    callbacks = [
        keras.callbacks.EarlyStopping(monitor="val_loss", patience=2, restore_best_weights=True),
    ]

    history = model.fit(
        [X_train, Y_train_in],
        Y_train_out,
        sample_weight=W_train,
        validation_data=([X_optuna_val, Y_optuna_val_in], Y_optuna_val_out, W_optuna_val),
        epochs=OPTUNA_EPOCHS,
        batch_size=BATCH_SIZE,
        verbose=0,
        callbacks=callbacks,
    )

    return min(history.history["val_loss"])


use_optuna = True
if use_optuna:
    study = optuna.create_study(direction="minimize")
    study.optimize(objective, n_trials=OPTUNA_TRIALS)
    best_params = study.best_params
    best_params["decoder_units"] = best_params["encoder_units"]
else:
    best_params = {
        "embedding_dim": 128,
        "encoder_units": 256,
        "decoder_units": 256,
        "dropout": 0.2,
        "learning_rate": 1e-3,
    }

print("Best params:", best_params)

[I 2026-05-21 13:48:25,633] A new study created in memory with name: no-name-537a1a34-599e-4296-b1fe-e6610e7df5ac
[I 2026-05-21 13:55:10,720] Trial 0 finished with value: 4.346795558929443 and parameters: {'embedding_dim': 128, 'encoder_units': 128, 'dropout': 0.2738954340269641, 'learning_rate': 0.0005064581590027736}. Best is trial 0 with value: 4.346795558929443.
[I 2026-05-21 14:06:32,297] Trial 1 finished with value: 3.1269149780273438 and parameters: {'embedding_dim': 256, 'encoder_units': 256, 'dropout': 0.3353422431096115, 'learning_rate': 0.001905473024868375}. Best is trial 1 with value: 3.1269149780273438.
[I 2026-05-21 14:19:48,584] Trial 2 finished with value: 4.411223888397217 and parameters: {'embedding_dim': 256, 'encoder_units': 256, 'dropout': 0.26542781821689027, 'learning_rate': 0.0002276370811712411}. Best is trial 1 with value: 3.1269149780273438.


Best params: {'embedding_dim': 256, 'encoder_units': 256, 'dropout': 0.3353422431096115, 'learning_rate': 0.001905473024868375, 'decoder_units': 256}


In [15]:
final_model = build_seq2seq_attention_model(
    source_vocab_size,
    target_vocab_size,
    **best_params,
)

callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=1, min_lr=1e-5),
]

history = final_model.fit(
    [X_train, Y_train_in],
    Y_train_out,
    sample_weight=W_train,
    validation_data=([X_optuna_val, Y_optuna_val_in], Y_optuna_val_out, W_optuna_val),
    epochs=FINAL_EPOCHS,
    batch_size=BATCH_SIZE,
    verbose=1,
    callbacks=callbacks,
)

train_loss = final_model.evaluate([X_train, Y_train_in], Y_train_out, sample_weight=W_train, verbose=0)
optuna_val_loss = final_model.evaluate([X_optuna_val, Y_optuna_val_in], Y_optuna_val_out, sample_weight=W_optuna_val, verbose=0)

encoder_model, decoder_model = build_inference_models(final_model)
decode_sequence = make_greedy_decoder(encoder_model, decoder_model, target_vectorizer)


def strip_special_tokens(text):
    return text.replace(START_TOKEN, "").replace(END_TOKEN, "").strip()



def decode_corpus(source_texts):
    return [decode_sequence(source_text) for source_text in source_texts]



def evaluate_translation_subset(source_texts, reference_texts, subset_size=200):
    limit = min(subset_size, len(source_texts), len(reference_texts))
    subset_sources = source_texts[:limit]
    subset_references = [strip_special_tokens(reference_text) for reference_text in reference_texts[:limit]]
    predictions = decode_corpus(subset_sources)
    sacrebleu_score = sacrebleu.corpus_bleu(predictions, [subset_references]).score
    chrfpp_score = sacrebleu.corpus_chrf(predictions, [subset_references], word_order=2).score
    meteor_values = [nltk_meteor_score([reference.split()], prediction.split()) for prediction, reference in zip(predictions, subset_references)]
    meteor_value = float(np.mean(meteor_values))
    return {
        "SacreBLEU": sacrebleu_score,
        "chrF++": chrfpp_score,
        "METEOR": meteor_value,
        "predictions": predictions,
        "references": subset_references,
        "subset_size": limit,
    }


val_metrics = evaluate_translation_subset(official_val_sources, official_val_targets, subset_size=200)
test_metrics = evaluate_translation_subset(official_test_sources, official_test_targets, subset_size=200)

print(f"Train loss: {train_loss:.4f}")
print(f"Optuna val loss: {optuna_val_loss:.4f}")
print(f"Validation metrics on {val_metrics['subset_size']} samples:")
print(f"  SacreBLEU: {val_metrics['SacreBLEU']:.2f}")
print(f"  chrF++:    {val_metrics['chrF++']:.2f}")
print(f"  METEOR:    {val_metrics['METEOR']:.4f}")
print(f"Test metrics on {test_metrics['subset_size']} samples:")
print(f"  SacreBLEU: {test_metrics['SacreBLEU']:.2f}")
print(f"  chrF++:    {test_metrics['chrF++']:.2f}")
print(f"  METEOR:    {test_metrics['METEOR']:.4f}")

sample_indices = np.linspace(0, test_metrics['subset_size'] - 1, num=min(5, test_metrics['subset_size']), dtype=int)
for index in sample_indices:
    source_sentence = official_test_sources[index]
    reference_sentence = strip_special_tokens(official_test_targets[index])
    predicted_sentence = test_metrics["predictions"][index]
    print("\nFR:", source_sentence)
    print("REF:", reference_sentence)
    print("PRD:", predicted_sentence)

Epoch 1/6
259/259 ━━━━━━━━━━━━━━━━━━━━ 358s 1s/step - loss: 4.5212 - val_loss: 3.7520 - learning_rate: 0.0019
Epoch 2/6
259/259 ━━━━━━━━━━━━━━━━━━━━ 358s 1s/step - loss: 3.4207 - val_loss: 3.1890 - learning_rate: 0.0019
Epoch 3/6
259/259 ━━━━━━━━━━━━━━━━━━━━ 356s 1s/step - loss: 2.7299 - val_loss: 2.8103 - learning_rate: 0.0019
Epoch 4/6
259/259 ━━━━━━━━━━━━━━━━━━━━ 345s 1s/step - loss: 2.1449 - val_loss: 2.6205 - learning_rate: 0.0019
Epoch 5/6
259/259 ━━━━━━━━━━━━━━━━━━━━ 346s 1s/step - loss: 1.6732 - val_loss: 2.5397 - learning_rate: 0.0019
Epoch 6/6
259/259 ━━━━━━━━━━━━━━━━━━━━ 340s 1s/step - loss: 1.3032 - val_loss: 2.4870 - learning_rate: 0.0019
Train loss: 0.7549
Optuna val loss: 2.4830
Validation metrics on 200 samples:
  SacreBLEU: 5.60
  chrF++:    25.24
  METEOR:    0.2568
Test metrics on 200 samples:
  SacreBLEU: 4.06
  chrF++:    22.45
  METEOR:    0.2218

FR: Je lui ai montré ma chambre.
REF: I showed her my room.
PRD: i was my room to my room

FR: Je ferai une exception,